# Notebook 03 — Entrenamiento: Satelital → Boceto Cartográfico (A→B)

> **Blog Técnico** · Pix2Pix: Traducción `AtoB` — satelite a mapa de carreteras

Este notebook entrena el modelo en la dirección **A→B**: dada una fotografía aérea/satelital, el generador aprende a sintetizar el mapa de carreteras equivalente.

**Configuración de entrenamiento (paper Pix2Pix):**

| Hiperparámetro | Valor | Justificación |
|---|---|---|
| `batch_size` | 1 | InstanceNorm funciona con batch=1; BatchNorm no |
| `lr` | 0.0002 | Estándar para Adam en GANs (DCGAN paper) |
| `beta1` | 0.5 | Reduce momentum en Adam para estabilizar GANs |
| `lambda_l1` | 100 | L1 domina (~97%) sobre GAN loss |
| `n_epochs` | 100 | LR constante |
| `n_epochs_decay` | 100 | LR decae linealmente hasta 0 |
| `use_amp` | True | Mixed Precision: -40% VRAM en T4 |
| `grad_accum_steps` | 4 | Batch efectivo = 4 sin más VRAM |

**Scheduler de LR lineal:**
$$\text{lr}(e) = \begin{cases} \text{lr}_0 & e \leq n_\text{epochs} \\ \text{lr}_0 \cdot \left(1 - \dfrac{e - n_\text{epochs}}{n_\text{epochs\_decay}}\right) & e > n_\text{epochs} \end{cases}$$

In [ ]:
import sys, os
from pathlib import Path

PROYECTO = Path('..').resolve() if Path('../src').exists() else Path('.')
os.chdir(PROYECTO)
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))

import torch
from train import config_desde_args, bucle_entrenamiento, generar_nombre_experimento

print(f"PyTorch  : {torch.__version__}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No disponible'}")
print(f"Proyecto : {PROYECTO}")

## 1. Configuración del Experimento

Modifica los parámetros a continuación según tu entorno. Para **Colab T4 gratuita**, la configuración por defecto es óptima.

> **Tip para sesiones cortas:** Si Colab puede reiniciarse, usa `--reanudar` para continuar desde el último checkpoint automáticamente.

In [ ]:
# ── Parámetros del experimento ────────────────────────────────────────────────
# Modifica esta lista para cambiar la configuración
ARGUMENTOS = [
    '--datos',           'data/processed',
    '--direction',       'AtoB',
    '--epocas',          '100',
    '--epocas_decay',    '100',
    '--batch',           '1',
    '--lr',              '0.0002',
    '--beta1',           '0.5',
    '--lambda_l1',       '100',
    '--gan_mode',        'lsgan',
    '--grad_accum',      '4',
    '--amp',                            # activar Mixed Precision (requiere CUDA)
    '--frecuencia_ckpt', '10',
    '--frecuencia_muestra', '5',
    '--workers',         '2',
    # '--reanudar',                     # descomentar para reanudar desde checkpoint
    # '--nombre_exp',    'sat2sketch_v1',  # nombre personalizado
]

config, args = config_desde_args(ARGUMENTOS)
nombre_exp = generar_nombre_experimento(config, args)

config.imprimir_resumen()
print(f"  Nombre experimento : {nombre_exp}")
print(f"  Checkpoints en     : checkpoints/{nombre_exp}/")
print(f"  Resultados en      : results/{nombre_exp}/")

## 2. Verificar Dataset y Estimar Tiempo

Antes de lanzar el entrenamiento largo, verificamos que el dataset esté bien cargado y estimamos el tiempo total.

In [ ]:
from src.data.dataset_loader import DatasetParesSideBySide, crear_dataloader
import time

# Cargar dataset y medir velocidad
dataset_train = DatasetParesSideBySide(
    str(Path('data/processed') / 'train'),
    direction='AtoB', modo='train',
)
dl_train = crear_dataloader(dataset_train, batch_size=1, modo='train',
                            num_workers=2, shuffle=True)

# Medir velocidad real del DataLoader
tiempos = []
it = iter(dl_train)
for _ in range(10):
    t0 = time.time()
    next(it)
    tiempos.append(time.time() - t0)

ms_por_batch = sum(tiempos) / len(tiempos) * 1000
n_batches    = len(dl_train)
segs_por_ep  = ms_por_batch * n_batches / 1000
total_epocas = config.n_epochs + config.n_epochs_decay

print(f"Dataset train  : {len(dataset_train)} pares")
print(f"Batches/época  : {n_batches}")
print(f"ms/batch       : {ms_por_batch:.1f} ms")
print(f"Tiempo/época   : ~{segs_por_ep/60:.1f} min")
print(f"Tiempo total   : ~{segs_por_ep * total_epocas / 3600:.1f} h ({total_epocas} épocas)")
print()
print("[!] Tiempo estimado solo de carga de datos. El backward pass")
print("    puede añadir 2-5x según la GPU.")

## 3. Lanzar Entrenamiento

> ⚠️ **Ejecución larga**: ~8-13 horas en T4 para 200 épocas completas.  
> Los checkpoints se guardan automáticamente cada 10 épocas en `checkpoints/`.  
> Para reanudar si Colab se reinicia, descomenta `--reanudar` en la celda de configuración.

El bucle de entrenamiento alterna:
1. `backward_D`: actualizar discriminador (pares real y falso)
2. `backward_G`: actualizar generador (engañar a D + L1 con el real)

In [ ]:
# ── LANZAR ENTRENAMIENTO ──────────────────────────────────────────────────────
# Esta celda lanza el entrenamiento completo.
# Los logs aparecerán en tiempo real abajo.

entrenador = bucle_entrenamiento(
    config=config,
    nombre_exp=nombre_exp,
    args_extra=args,
)

print("\n[OK] Entrenamiento AtoB completado.")
print(f"     Checkpoints en: checkpoints/{nombre_exp}/")
print(f"     Muestras en   : results/{nombre_exp}/")
print("     Siguiente: 05_inference_demo.ipynb para probar el modelo.")

## 4. Monitoreo Durante el Entrenamiento

While training runs, you can inspect the saved sample images in `results/` to track progress visually, and read the loss log in `logs/`.

### Señales de alerta en las pérdidas:

| Síntoma | Causa probable | Solución |
|---------|---------------|----------|
| `loss_D ≈ 0` | D demasiado fuerte, G no mejora | Reducir `lr_D` o añadir label smoothing |
| `loss_D ≈ 0.5` | Equilibrio ideal | ✓ Continuar |
| `loss_G_GAN → 0` | Mode collapse (G ignora la condición) | Verificar que D recibe el par (x, y) |
| `loss_G_L1 sin descender` | LR demasiado bajo o dataset mal cargado | Verificar normalización |
| NaN en pérdidas | Overflow numérico con AMP | Usar `--no_amp` o reducir `lr` |

In [ ]:
# Visualizar la última muestra guardada durante el entrenamiento
import glob
import matplotlib.pyplot as plt
from PIL import Image as PILImage

muestras = sorted(glob.glob(f'results/{nombre_exp}/muestra_epoca_*.png'))
if muestras:
    ultima = muestras[-1]
    with PILImage.open(ultima) as img:
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'Última muestra: {Path(ultima).name}', fontsize=11)
        plt.tight_layout()
        plt.show()
    print(f"Épocas completadas: {len(muestras) * config.frecuencia_muestra}")
else:
    print("Aún no hay muestras guardadas. El entrenamiento debe completar al menos 1 época.")